[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch14-ml-ops/04-lech-va-phat-hanh.ipynb)

# Lệch giữa huấn luyện và phục vụ, rồi phát hành có kiểm soát

Một ngộ nhận cuối chương 14 cho rằng môi trường huấn luyện và phục vụ tự động nhất quán một khi pipeline
đã dựng xong. Thực tế, lệch giữa huấn luyện và phục vụ (training-serving skew) sinh ra từ những khác biệt
nhỏ trong logic tiền xử lý, cách xử lý múi giờ hay phiên bản thư viện. Mô hình không đổi một dòng mã nào,
vậy mà độ chính xác giảm. Nửa đầu sổ tay này gây ra từng loại lệch trong bảng 7 của chương và đo cái giá
của nó; nửa sau đi theo mô hình ra sản xuất: một chuỗi mô hình sửa lỗi cho nhau, một kiểm thử A/B, một đợt
triển khai thăm dò (canary deployment) và ngân sách độ trễ của một yêu cầu.

**Bạn sẽ làm:**
1. gây ba loại lệch của bảng 7 trên `load_digits` (chuẩn hoá, giá trị thiếu, một tham số mặc định của thư
   viện) và đo độ chính xác;
2. dựng lại ca `user_session_length` (45 phút lúc huấn luyện, 12 phút lúc phục vụ) trên dữ liệu tổng hợp,
   chặn nó bằng một cổng kiểm định Kolmogorov-Smirnov trong CI, rồi tìm điểm mù của cổng ấy;
3. xem một đặc trưng tính sai mốc thời gian làm độ chính xác ngoại tuyến đẹp giả ra sao;
4. dựng một thác hiệu chỉnh: mô hình B sửa lỗi của mô hình A, rồi cải thiện A;
5. tính cỡ mẫu A/B của phương trình 9 và đo xem nhìn trộm kết quả làm tăng báo nhầm bao nhiêu;
6. so canary 5 và 30 phần trăm, tính ngân sách độ trễ của bảng 14, và kiểm số học của ca Oura.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import hashlib
import json

from scipy import ndimage, stats
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
XAM = "0.45"
KS_NGUONG = sach(0.1, nguon="KS 0.1")             # ngưỡng của bảng 19, điểm khởi đầu phải hiệu chuẩn

X16, y = load_digits(return_X_y=True)             # điểm ảnh nguyên từ 0 tới 16
thu_tu = np.random.default_rng(SEED).permutation(len(X16))
hl, pv = thu_tu[:1078], thu_tu[1078:]             # ảnh huấn luyện và ảnh "phục vụ"


def dung(mo_hinh, dac_trung, chi_so):
    return 100 * float((mo_hinh.predict(dac_trung) == y[chi_so]).mean())


def cong_ks(hl_dt, pv_dt, nguong=KS_NGUONG):
    """Cổng CI theo tinh thần listing 2: từ chối khi có đặc trưng nào có thống kê KS vượt ngưỡng."""
    ks = [stats.ks_2samp(hl_dt[:, j], pv_dt[:, j]).statistic for j in range(hl_dt.shape[1])]
    j = int(np.argmax(ks))
    return ks[j] <= nguong, j, float(ks[j])


print(f"{len(hl)} ảnh huấn luyện, {len(pv)} ảnh phục vụ")

## 1. Ba loại lệch của bảng 7, đo bằng độ chính xác

Bảng 7 liệt kê bốn loại lệch. Ô dưới gây ra ba loại trên cùng một mô hình hồi quy logistic, mỗi lần chỉ
đổi đường phục vụ, còn mô hình và đường huấn luyện giữ nguyên:

* **tiền xử lý đặc trưng**: đường huấn luyện chia điểm ảnh cho 16, giá trị lớn nhất của bộ dữ liệu; đường
  phục vụ chia cho 255, như với ảnh 8 bit;
* **xử lý dữ liệu thiếu**: 20 phần trăm điểm ảnh bị mất (giả định của sổ tay); huấn luyện điền giá trị
  trung bình, phục vụ điền 0, đúng ví dụ của bảng 7;
* **phiên bản thư viện**: ảnh đến hơi nghiêng một góc đã biết, và cả hai đường xoay ngược lại bằng
  `scipy.ndimage.rotate`. Đường huấn luyện ghi rõ `mode="grid-constant"`; đường phục vụ để mặc định,
  là `mode="constant"`. Cùng một hàm, cùng một góc, khác một tham số không ai đọc tới.

**Dự đoán:** loại nào đắt nhất? Có loại nào làm mô hình gần như đoán một lớp duy nhất không?

In [ ]:
VI_DU = sach(99.2, nguon="99.2 phần trăm")
print(f"ví dụ của sách: một mô hình gian lận đạt {VI_DU} phần trăm độ chuẩn xác khi phát triển,"
      " rồi đoán mọi giao dịch là hợp lệ trên một máy chủ chạy phiên bản thư viện khác.\n")

ket_qua_lech = {}

# (a) chuẩn hoá bằng hai hằng số khác nhau
mo_hinh = LogisticRegression(max_iter=2000).fit(X16[hl] / 16, y[hl])
du_doan_255 = mo_hinh.predict(X16[pv] / 255)
ket_qua_lech["chia 16 / chia 255"] = (dung(mo_hinh, X16[pv] / 16, pv), dung(mo_hinh, X16[pv] / 255, pv))
lop_nhieu = np.bincount(du_doan_255, minlength=10).argmax()
print(f"chia cho 255: {np.mean(du_doan_255 == lop_nhieu):.0%} ảnh bị đoán là chữ số {lop_nhieu}")

# (b) giá trị thiếu: điền trung bình lúc huấn luyện, điền 0 lúc phục vụ
X = X16 / 16
thieu = np.random.default_rng(SEED).random(X.shape) < 0.20
Xt = np.where(thieu, np.nan, X)
tb_hl = np.nanmean(Xt[hl], axis=0)
dien_tb = lambda A: np.where(np.isnan(A), tb_hl, A)
dien_0 = lambda A: np.nan_to_num(A, nan=0.0)
mo_hinh_b = LogisticRegression(max_iter=2000).fit(dien_tb(Xt[hl]), y[hl])
ket_qua_lech["điền trung bình / điền 0"] = (dung(mo_hinh_b, dien_tb(Xt[pv]), pv), dung(mo_hinh_b, dien_0(Xt[pv]), pv))


# (c) một tham số mặc định của thư viện
def xoay(anh, goc, **kw):
    return np.clip(np.stack([ndimage.rotate(a.reshape(8, 8), g, reshape=False, order=1, **kw).ravel()
                             for a, g in zip(anh, goc)]), 0, 1)


r = np.random.default_rng(SEED)
goc_hl, goc_pv = r.uniform(-10, 10, len(hl)), r.uniform(-10, 10, len(pv))
nghieng_hl = xoay(X[hl], goc_hl, mode="grid-constant")       # ảnh đến hơi nghiêng
nghieng_pv = xoay(X[pv], goc_pv, mode="grid-constant")
CAU_HINH_HL = {"ham": "scipy.ndimage.rotate", "order": 1, "mode": "grid-constant"}
CAU_HINH_PV = {"ham": "scipy.ndimage.rotate", "order": 1}   # để mặc định
dt_hl = xoay(nghieng_hl, -goc_hl, mode="grid-constant")
mo_hinh_c = LogisticRegression(max_iter=2000).fit(dt_hl, y[hl])
dt_pv_dung = xoay(nghieng_pv, -goc_pv, mode="grid-constant")
dt_pv_lech = xoay(nghieng_pv, -goc_pv)
ket_qua_lech["mode ghi rõ / mode mặc định"] = (dung(mo_hinh_c, dt_pv_dung, pv), dung(mo_hinh_c, dt_pv_lech, pv))

for ten, (a, b) in ket_qua_lech.items():
    do_tren_may(f"{ten}: đường đúng", a, "%")
    do_tren_may(f"{ten}: đường lệch", b, "%")

Mỗi loại có cách phát hiện riêng trong bảng 7: so phân phối đặc trưng, kiểm tra giản đồ với quy tắc rõ
ràng cho giá trị rỗng, và so mã băm (hash) của môi trường. Ô dưới chạy cả hai cách tự động: cổng KS trên
phân phối từng điểm ảnh của lưu lượng phục vụ, và so mã băm cấu hình tiền xử lý của hai đường.

In [ ]:
def ma_bam(cau_hinh):
    return hashlib.sha256(json.dumps(cau_hinh, sort_keys=True).encode()).hexdigest()[:12]


cap = {"chia 16 / chia 255": (X16[hl] / 16, X16[pv] / 16, X16[pv] / 255),
       "điền trung bình / điền 0": (dien_tb(Xt[hl]), dien_tb(Xt[pv]), dien_0(Xt[pv])),
       "mode ghi rõ / mode mặc định": (dt_hl, dt_pv_dung, dt_pv_lech)}
for ten, (a_hl, a_dung, a_lech) in cap.items():
    qua_dung, _, ks_dung = cong_ks(a_hl, a_dung)
    qua_lech, j, ks_lech = cong_ks(a_hl, a_lech)
    print(f"{ten:<28} đường đúng: KS lớn nhất {ks_dung:.3f} ({'qua' if qua_dung else 'chặn'})"
          f" · đường lệch: {ks_lech:.3f} ở điểm ảnh {j} ({'qua' if qua_lech else 'chặn'})")
print(f"\nmã băm cấu hình xoay: huấn luyện {ma_bam(CAU_HINH_HL)}, phục vụ {ma_bam(CAU_HINH_PV)}"
      f" → {'khớp' if ma_bam(CAU_HINH_HL) == ma_bam(CAU_HINH_PV) else 'KHÁC, chặn triển khai'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
vi_tri = np.arange(len(ket_qua_lech))
ax.bar(vi_tri - 0.18, [a for a, _ in ket_qua_lech.values()], 0.34, color=MAU[0], label="đường phục vụ đúng")
ax.bar(vi_tri + 0.18, [b for _, b in ket_qua_lech.values()], 0.34, color=MAU[1], label="đường phục vụ lệch")
for i, (a, b) in enumerate(ket_qua_lech.values()):
    ax.text(i - 0.18, a + 1.5, f"{a:.0f}", ha="center", fontsize=8)
    ax.text(i + 0.18, b + 1.5, f"{b:.0f}", ha="center", fontsize=8)
ax.set_xticks(vi_tri, list(ket_qua_lech))
ax.set_ylim(0, 112)
ax.set_ylabel("độ chính xác (%)")
ax.set_title("Cùng một mô hình, hai đường phục vụ (đo trên máy này)")
ax.legend(loc="upper right", fontsize=8, frameon=False, ncol=2)
plt.tight_layout()
plt.show()

Trong ba ca, mô hình không hề thay đổi. Ca chia cho 255 là phiên bản chữ số của ví dụ gian lận: đầu vào
nhỏ đi gần 16 lần, các điểm số của mô hình bị số hạng chặn chi phối, nên dự đoán dồn về một hay vài lớp. Cổng KS
bắt được cả ba vì mỗi lỗi làm đổi phân phối của ít nhất một điểm ảnh, và mã băm cấu hình bắt được ca thứ ba
trước khi có một yêu cầu nào được phục vụ.

## 2. Ca `user_session_length`: 45 phút so với 12 phút

Ví dụ của chương: một hệ gợi ý giảm 8 phần trăm độ chính xác một tháng sau khi triển khai, không đổi một
dòng mã mô hình nào. So phân phối đặc trưng thì thấy `user_session_length` có trung bình 45 phút lúc huấn
luyện nhưng 12 phút lúc phục vụ. Pipeline huấn luyện tính thời gian theo đồng hồ từ sự kiện đầu tới sự kiện
cuối của phiên; đường phục vụ chỉ đếm thời gian hoạt động sau khi bỏ các quãng nghỉ.

Ô dưới tạo 30,000 phiên tổng hợp từ chuỗi khoảng cách giữa các sự kiện. Một khoảng dài hơn 5 phút được
coi là nghỉ. Khả năng mua phụ thuộc vào thời gian hoạt động và số lần nhấp; số trang xem gần như không
liên quan. Mọi tham số là giả định của sổ tay, chọn để hai trung bình gần với ví dụ của sách.

In [ ]:
NGHI = 5.0                                         # phút; giả định


def tao_phien(n, r):
    """Mỗi phiên là một mảng khoảng cách (phút) giữa các sự kiện liên tiếp."""
    so_khoang = 1 + r.poisson(r.gamma(2.0, 7.0, n))
    phien = []
    for k in so_khoang:
        hoat_dong = r.random(k) < 0.85
        phien.append(np.where(hoat_dong, np.minimum(r.exponential(1.0, k), NGHI), NGHI + r.exponential(9.5, k)))
    return phien


def theo_dong_ho(phien):
    """Định nghĩa của pipeline huấn luyện: từ sự kiện đầu tới sự kiện cuối."""
    return np.array([g.sum() for g in phien])


def theo_hoat_dong(phien):
    """Định nghĩa của đường phục vụ: bỏ các quãng nghỉ."""
    return np.array([g[g <= NGHI].sum() for g in phien])


r = np.random.default_rng(SEED)
phien = tao_phien(30_000, r)
hoat = theo_hoat_dong(phien)
so_nhap = r.poisson(3, len(phien))
so_trang = r.poisson(3, len(phien))                # cùng phân phối với số lần nhấp, vai trò khác hẳn
logit = -3.0 + 0.25 * hoat + 0.5 * (so_nhap - 3) + 0.05 * (so_trang - 3)
mua = r.random(len(phien)) < 1 / (1 + np.exp(-logit))
tap_hl, tap_pv = np.arange(20_000), np.arange(20_000, 30_000)

dh = theo_dong_ho(phien)
print(f"sách: {sach(45, nguon='45 phút · 12 phút · 8 phần trăm · một tháng')} phút lúc huấn luyện,"
      f" {sach(12, nguon='45 phút · 12 phút · 8 phần trăm · một tháng')} phút lúc phục vụ")
print(f"dữ liệu tổng hợp: theo đồng hồ {dh.mean():.1f} phút, theo hoạt động {hoat.mean():.1f} phút;"
      f" tỉ lệ mua {mua.mean():.0%}")


def dac_trung(dinh_nghia, cot=("thoi_luong", "so_nhap", "so_trang")):
    """Ghép các cột đặc trưng theo thứ tự cho trước."""
    cac_cot = {"thoi_luong": dinh_nghia(phien), "so_nhap": so_nhap, "so_trang": so_trang}
    return np.column_stack([cac_cot[c] for c in cot])


F_hl = dac_trung(theo_dong_ho)
mo_hinh_p = LogisticRegression(max_iter=2000).fit(F_hl[tap_hl], mua[tap_hl])
dung_p = lambda m, F: 100 * float((m.predict(F[tap_pv]) == mua[tap_pv]).mean())
F_pv_lech = dac_trung(theo_hoat_dong)
do_tren_may("đúng định nghĩa lúc huấn luyện", dung_p(mo_hinh_p, F_hl), "%")
do_tren_may("phục vụ bằng định nghĩa thứ hai", dung_p(mo_hinh_p, F_pv_lech), "%")
print(f"tỉ lệ phiên được đoán là mua: {mo_hinh_p.predict(F_hl[tap_pv]).mean():.0%} → {mo_hinh_p.predict(F_pv_lech[tap_pv]).mean():.0%}")
qua, j, ks = cong_ks(F_hl[tap_hl], F_pv_lech[tap_pv])
print(f"cổng KS: thống kê lớn nhất {ks:.3f} ở cột {j} → {'qua' if qua else 'chặn triển khai'}")

Mô hình học một ngưỡng trên thời gian theo đồng hồ, rồi nhận những con số nhỏ hơn chừng bốn lần, nên số
phiên vượt ngưỡng tụt hẳn. Cổng KS thấy ngay, vì hai định nghĩa cho hai phân phối rất khác nhau.

Cách sửa của sách là định nghĩa `session_length` **một lần**, trong kho đặc trưng (feature store), và để cả
hai đường gọi cùng định nghĩa đó. Nhưng sách cũng nói tính ngang bằng (parity) vẫn phải được kiểm tra. Ô
dưới làm cả hai: chọn định nghĩa mà đường phục vụ tính được, huấn luyện lại trên nó, và kiểm ngang bằng
trên **cùng những phiên** chạy qua cả hai đường. Rồi nó thử một lỗi mà cổng KS không thể thấy: đường phục
vụ đảo thứ tự hai cột `so_nhap` và `so_trang`, hai đặc trưng có cùng phân phối.

**Dự đoán:** cổng KS có chặn được lỗi đảo cột không?

In [ ]:
DINH_NGHIA_CHUNG = theo_hoat_dong                  # một định nghĩa, cả hai đường cùng gọi
F_chung = dac_trung(DINH_NGHIA_CHUNG)
mo_hinh_chung = LogisticRegression(max_iter=2000).fit(F_chung[tap_hl], mua[tap_hl])
do_tren_may("định nghĩa chung, huấn luyện lại", dung_p(mo_hinh_chung, F_chung), "%")


def kiem_ngang_bang(duong_hl, duong_pv, chi_so):
    """So từng giá trị của cùng những dòng qua hai đường. Trả về tỉ lệ dòng khác nhau."""
    return float(np.mean(np.any(duong_hl[chi_so] != duong_pv[chi_so], axis=1)))


mau_kiem = np.random.default_rng(SEED).choice(tap_pv, 500, replace=False)
assert kiem_ngang_bang(F_chung, dac_trung(DINH_NGHIA_CHUNG), mau_kiem) == 0.0    # cùng định nghĩa thì khớp tuyệt đối

F_dao = dac_trung(DINH_NGHIA_CHUNG, cot=("thoi_luong", "so_trang", "so_nhap"))   # lỗi giản đồ: đảo hai cột
qua, j, ks = cong_ks(F_chung[tap_hl], F_dao[tap_pv])
print(f"đảo hai cột · cổng KS: thống kê lớn nhất {ks:.3f} → {'qua' if qua else 'chặn'}")
lech_dong = kiem_ngang_bang(F_chung, F_dao, mau_kiem)
print(f"đảo hai cột · kiểm ngang bằng trên {len(mau_kiem)} phiên: {lech_dong:.0%} dòng khác nhau → "
      f"{'qua' if lech_dong == 0 else 'chặn'}")
do_tren_may("định nghĩa chung, hai cột bị đảo", dung_p(mo_hinh_chung, F_dao), "%")

Hai cột có cùng phân phối nên thống kê KS của từng cột chỉ là nhiễu lấy mẫu, và cổng cho qua. Đúng điều sách
cảnh báo: một thống kê vượt qua không chứng minh được sự ngang bằng về giản đồ, ngữ nghĩa hay mốc thời
gian. Phép kiểm ngang bằng trên cùng những dòng thì bắt được ngay, vì nó so giá trị chứ không so phân phối.

## 3. Đúng theo mốc thời gian

Kho đặc trưng của chương lấy đặc trưng lịch sử "đúng theo mốc thời gian" (point-in-time correctness) cho
huấn luyện: giá trị của một đặc trưng phải là giá trị **tại lúc dự đoán**, không phải giá trị tính lại sau
đó. Bảng 7 gọi lỗi ngược lại là đặc trưng tính với mốc cắt thời gian khác nhau.

Ô dưới thêm một đặc trưng "số lần mua trước đây" vào dữ liệu phiên. Bản tính sai được tính lại vào cuối
ngày, nên nó gồm cả lần mua của chính phiên đang được dự đoán. Lúc phục vụ, lần mua ấy chưa xảy ra.

**Dự đoán:** mô hình học trên bản tính sai đạt bao nhiêu khi kiểm định ngoại tuyến, và bao nhiêu khi phục vụ?

In [ ]:
mua_truoc = r.poisson(1.0 + 0.5 * (so_nhap > 3))
logit2 = logit + 0.4 * mua_truoc
mua2 = r.random(len(phien)) < 1 / (1 + np.exp(-logit2))
cuoi_ngay = mua_truoc + mua2                       # tính lại cuối ngày: gồm cả lần mua đang dự đoán
G_sai = np.column_stack([F_chung, cuoi_ngay])
G_dung = np.column_stack([F_chung, mua_truoc])     # đúng theo mốc thời gian, cũng là thứ đường phục vụ có
dung2 = lambda m, G: 100 * float((m.predict(G[tap_pv]) == mua2[tap_pv]).mean())
m_sai = LogisticRegression(max_iter=2000).fit(G_sai[tap_hl], mua2[tap_hl])
m_dung = LogisticRegression(max_iter=2000).fit(G_dung[tap_hl], mua2[tap_hl])
ket_qua_moc = {"bản tính sai, kiểm định ngoại tuyến": dung2(m_sai, G_sai),
               "bản tính sai, khi phục vụ": dung2(m_sai, G_dung),
               "đúng theo mốc thời gian, khi phục vụ": dung2(m_dung, G_dung)}
for ten, v in ket_qua_moc.items():
    do_tren_may(ten, v, "%")
print(f"đoán luôn lớp đông nhất: {100 * max(mua2[tap_pv].mean(), 1 - mua2[tap_pv].mean()):.1f} %")

Con số ngoại tuyến của bản tính sai đo một bài toán dễ hơn bài toán thật, vì đặc trưng đã chứa một phần câu
trả lời. Khi phục vụ, phần ấy biến mất, và mô hình đã học dựa vào nó. Không phép so phân phối nào trên dữ
liệu phục vụ chỉ ra được điều này, vì chính dữ liệu huấn luyện mới là chỗ sai; chỉ kiểm tra mốc thời gian
trong pipeline đặc trưng, như bảng 7 khuyên, mới bắt được.

## 4. Thác hiệu chỉnh

Hình 4 của chương vẽ một chuỗi mô hình phụ thuộc: mô hình B được huấn luyện để sửa lỗi còn sót của mô hình A.
Chuỗi đứng vững cho tới khi A thay đổi; khi A được huấn luyện lại để sửa một lỗi, mọi mô hình phía sau đã chỉnh
theo hành vi cũ của nó đều mất hiệu lực cùng lúc.

Ô dưới dựng một chuỗi hai mắt xích trên `load_digits`. A là một mô hình hồi quy logistic điều chuẩn mạnh, học
trên 500 ảnh. B nhận log xác suất của A trên 578 ảnh khác và học cách sửa A. Rồi nhóm "cải thiện" A bằng cách
giảm điều chuẩn, và giữ nguyên B.

**Dự đoán:** hệ thống A mới cộng B cũ tốt hơn hay kém hơn A mới một mình?

In [ ]:
X = X16 / 16
hl_a, hl_b = hl[:500], hl[500:]
log_xs = lambda m, A: np.log(m.predict_proba(A) + 1e-9)
A_cu = LogisticRegression(max_iter=2000, C=0.02).fit(X[hl_a], y[hl_a])
B_cu = LogisticRegression(max_iter=2000).fit(log_xs(A_cu, X[hl_b]), y[hl_b])
A_moi = LogisticRegression(max_iter=2000, C=1.0).fit(X[hl_a], y[hl_a])
B_moi = LogisticRegression(max_iter=2000).fit(log_xs(A_moi, X[hl_b]), y[hl_b])
chuoi = {"A cũ": dung(A_cu, X[pv], pv),
         "A cũ + B cũ": dung(B_cu, log_xs(A_cu, X[pv]), pv),
         "A mới": dung(A_moi, X[pv], pv),
         "A mới + B cũ": dung(B_cu, log_xs(A_moi, X[pv]), pv),
         "A mới + B học lại": dung(B_moi, log_xs(A_moi, X[pv]), pv)}
for ten, v in chuoi.items():
    do_tren_may(ten, v, "%")
print(f"\nsau khi cải thiện A mà giữ B: hệ thống {'kém hơn' if chuoi['A mới + B cũ'] < chuoi['A mới'] else 'không kém'}"
      f" A mới một mình {abs(chuoi['A mới'] - chuoi['A mới + B cũ']):.1f} điểm,"
      f" và {'kém hơn' if chuoi['A mới + B cũ'] < chuoi['A cũ + B cũ'] else 'không kém'} hệ thống trước khi sửa")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
mau_cot = [MAU[0], MAU[0], MAU[2], MAU[1], MAU[2]]
ax.barh(list(chuoi)[::-1], list(chuoi.values())[::-1], color=mau_cot[::-1], height=0.6)
for i, v in enumerate(list(chuoi.values())[::-1]):
    ax.text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=8)
ax.set_xlim(50, 102)
ax.set_xlabel("độ chính xác (%)")
ax.set_title("Thác hiệu chỉnh: B học theo A cũ (đo trên máy này)")
plt.tight_layout()
plt.show()

B không học chữ số; nó học **thói quen của A cũ**: A cũ điều chuẩn mạnh nên xác suất của nó dàn đều, và B
học cách đọc những xác suất dàn đều ấy. A mới tự tin hơn nhiều, cùng một log xác suất giờ mang nghĩa khác,
và B đọc sai. Phụ thuộc này đi qua dữ liệu, không qua một giao diện mã nào, nên công cụ phân tích phụ thuộc
thông thường không thấy. Một thay đổi gần đầu chuỗi buộc phải học lại mọi thứ phía sau.

## 5. Phương trình 9: cỡ mẫu A/B, và cái giá của nhìn trộm

Với xấp xỉ chuẩn và phương sai chung, số người dùng cần cho mỗi nhánh là

$$n = \frac{2 (z_{\alpha/2} + z_\beta)^2 \sigma^2}{\delta^2}$$

với $\delta$ là hiệu ứng nhỏ nhất phát hiện được. Ví dụ của sách: độ tin cậy 95 phần trăm, lực kiểm định 80
phần trăm, tỉ lệ chuyển đổi 5 phần trăm và mức tăng tương đối 2 phần trăm (5 lên 5.1 phần trăm).

In [ ]:
TIN_CAY = sach(95, nguon="95 phần trăm · 80 phần trăm · 2 phần trăm tương đối") / 100
LUC = sach(80, nguon="95 phần trăm · 80 phần trăm") / 100
P_GOC = sach(5, nguon="5 → 5.1 phần trăm") / 100
P_MOI = sach(5.1, nguon="5 → 5.1 phần trăm") / 100
TANG_TUONG_DOI = sach(2, nguon="2 phần trăm tương đối") / 100
assert abs(P_GOC * (1 + TANG_TUONG_DOI) - P_MOI) < 1e-12
z_a, z_b = stats.norm.ppf(1 - (1 - TIN_CAY) / 2), stats.norm.ppf(LUC)
sigma2 = P_GOC * (1 - P_GOC)                       # phương sai của một biến Bernoulli ở tỉ lệ gốc


def n_moi_nhanh(delta, s2=sigma2):
    """Phương trình 9."""
    return 2 * (z_a + z_b) ** 2 * s2 / delta ** 2


def hieu_ung_nho_nhat(n, s2=sigma2):
    """Phương trình 9, giải ngược theo delta."""
    return np.sqrt(2 * (z_a + z_b) ** 2 * s2 / n)


theo_sach("người dùng mỗi nhánh", n_moi_nhanh(P_MOI - P_GOC), sach=745_644, nguon="khoảng 745,644", tol=1)
N_NHO = sach(25_000, nguon="25,000")
theo_sach("hiệu ứng nhỏ nhất ở 25,000 người mỗi nhánh (điểm phần trăm)", 100 * hieu_ung_nho_nhat(N_NHO),
          sach=0.5, nguon="khoảng 0.5 điểm phần trăm", tol=0.05)

Sách dặn chạy tới đúng cỡ mẫu đã đăng ký trước, và không dừng khi kết quả vừa có ý nghĩa thống kê, vì nhìn
trộm nhiều lần làm tăng tỉ lệ dương tính giả (false positive rate). Ô dưới mô phỏng 20,000 kiểm thử A/A: hai
nhánh giống hệt nhau ở 5 phần trăm, 25,000 người mỗi nhánh, chia thành tối đa 50 lần xem kết quả. Mọi kết luận
"có khác biệt" ở đây đều là báo nhầm.

**Dự đoán:** xem 20 lần và dừng ngay khi $|z| > 1.96$ thì báo nhầm bao nhiêu phần trăm?

In [ ]:
SO_PHEP_THU = 20_000
r_ab = np.random.default_rng(SEED)
LAN_XEM_MAX = 50
moi_lan = N_NHO // LAN_XEM_MAX
cong_a = r_ab.binomial(moi_lan, P_GOC, (SO_PHEP_THU, LAN_XEM_MAX)).cumsum(axis=1)
cong_b = r_ab.binomial(moi_lan, P_GOC, (SO_PHEP_THU, LAN_XEM_MAX)).cumsum(axis=1)


def z_tai(lan_xem, a=cong_a, b=cong_b):
    """Thống kê z của hiệu hai tỉ lệ tại những lần xem cho trước (chỉ số cột)."""
    n = moi_lan * (np.asarray(lan_xem) + 1)
    pa, pb = a[:, lan_xem] / n, b[:, lan_xem] / n
    pg = (pa + pb) / 2
    return (pb - pa) / np.sqrt(pg * (1 - pg) * 2 / n)


cac_so_lan = [1, 2, 5, 10, 20, 50]
bao_nham = {}
for k in cac_so_lan:
    lan_xem = np.round(np.linspace(LAN_XEM_MAX / k, LAN_XEM_MAX, k)).astype(int) - 1
    bao_nham[k] = float((np.abs(z_tai(lan_xem)) > z_a).any(axis=1).mean())
    print(f"xem {k:>2} lần, dừng khi |z| > {z_a:.2f}: báo nhầm {bao_nham[k]:.1%}")

# Một thiết kế tuần tự định trước: một ngưỡng chung cho 20 lần xem, hiệu chuẩn trên một nửa số kiểm thử,
# kiểm trên nửa còn lại.
lan_20 = np.round(np.linspace(LAN_XEM_MAX / 20, LAN_XEM_MAX, 20)).astype(int) - 1
z20 = np.abs(z_tai(lan_20))
nguong_tuan_tu = float(np.quantile(z20[: SO_PHEP_THU // 2].max(axis=1), 1 - (1 - TIN_CAY)))
kiem = float((z20[SO_PHEP_THU // 2:] > nguong_tuan_tu).any(axis=1).mean())
print(f"\nthiết kế tuần tự, 20 lần xem: ngưỡng |z| > {nguong_tuan_tu:.2f}, báo nhầm trên nửa kiểm tra {kiem:.1%}")

In [ ]:
cac_n = np.logspace(3.5, 6.5, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.plot(cac_n, 100 * hieu_ung_nho_nhat(cac_n), color=MAU[0], lw=2.2)
ax1.plot([N_NHO], [100 * hieu_ung_nho_nhat(N_NHO)], "o", color=MAU[1], ms=8, zorder=5)
ax1.annotate("25,000 người: khoảng 0.5 điểm", (N_NHO, 100 * hieu_ung_nho_nhat(N_NHO)), xytext=(12, 6),
             textcoords="offset points", fontsize=8)
ax1.plot([n_moi_nhanh(P_MOI - P_GOC)], [100 * (P_MOI - P_GOC)], "o", color=MAU[1], ms=8, zorder=5)
ax1.annotate("745,644 người: 0.1 điểm (5 lên 5.1 %)", (n_moi_nhanh(P_MOI - P_GOC), 100 * (P_MOI - P_GOC)),
             xytext=(-10, -22), textcoords="offset points", fontsize=8, ha="right")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("người dùng mỗi nhánh (thang log)")
ax1.set_ylabel("hiệu ứng nhỏ nhất phát hiện được\n(điểm phần trăm, thang log)")
ax1.set_title("Phương trình 9: gấp 4 lần người, hiệu ứng nhỏ đi một nửa")
ax2.plot(cac_so_lan, [100 * bao_nham[k] for k in cac_so_lan], "o-", color=MAU[1], lw=2)
ax2.axhline(100 * (1 - TIN_CAY), color=XAM, lw=0.9, ls="--")
ax2.text(6, 100 * (1 - TIN_CAY) - 2.8, "mức báo nhầm đã định: 5 %", fontsize=8, color="0.3")
ax2.set_xscale("log")
ax2.set_xticks(cac_so_lan, [str(k) for k in cac_so_lan])
ax2.set_ylim(0, None)
ax2.set_xlabel("số lần xem kết quả trong một kiểm thử A/A")
ax2.set_ylabel("kiểm thử báo có khác biệt (%)")
ax2.set_title("Nhìn trộm: dừng khi |z| > 1.96 lần đầu tiên")
plt.tight_layout()
plt.show()

Mỗi lần xem là một cơ hội để nhiễu vượt ngưỡng, nên càng xem nhiều, càng nhiều kiểm thử A/A tuyên bố một khác
biệt không tồn tại. Kiểm định tuần tự (sequential testing) không cấm xem; nó định trước một ngưỡng chặt hơn
cho mỗi lần xem để tổng báo nhầm vẫn là 5 phần trăm. Cái giá là phải có bằng chứng mạnh hơn ở mỗi lần xem.

## 6. Canary 5 phần trăm và 30 phần trăm, ngân sách độ trễ, và ca Oura

Sách kể hai chuyện về triển khai thăm dò. Một nền tảng thương mại điện tử đưa mô hình gợi ý mới tới 5 phần
trăm lưu lượng web rồi mới nâng dần. Và có những vấn đề chỉ lộ ra ở 30 phần trăm, không lộ ở 5 phần trăm.
Ô dưới tách hai lý do có thể. Thứ nhất là **bằng chứng**: phần lưu lượng nhỏ thì nhãn về chậm, nên một mức
giảm nhỏ cần nhiều giờ mới thấy. Thứ hai là **tải**: một phần phụ trợ của mô hình mới, ví dụ bộ đệm đặc
trưng của nó, có sức chứa giới hạn và chỉ quá tải khi phần lưu lượng đủ lớn. Lưu lượng, tốc độ nhãn và sức
chứa đều là giả định của sổ tay; hàng đợi được mô hình hoá như một hàng đợi M/M/1.

In [ ]:
CANARY_NHO = sach(5, nguon="30 phần trăm · 5 phần trăm") / 100
CANARY_LON = sach(30, nguon="30 phần trăm · 5 phần trăm") / 100
NHAN_MOI_GIO = 2_000                               # giả định: kết quả có nhãn mỗi giờ, toàn bộ lưu lượng
YEU_CAU_MOI_GIAY = 1_000                           # giả định: toàn bộ lưu lượng
SUC_CHUA = 320                                     # giả định: yêu cầu mỗi giây mà phụ trợ của mô hình mới chịu được


def n_mot_phia(p0, p1, alpha=0.05, luc=0.80):
    """Cỡ mẫu xấp xỉ chuẩn cho kiểm định một phía của một tỉ lệ, như sổ tay 03."""
    za, zb = stats.norm.ppf(1 - alpha), stats.norm.ppf(luc)
    return ((za * np.sqrt(p0 * (1 - p0)) + zb * np.sqrt(p1 * (1 - p1))) / (p0 - p1)) ** 2


can = n_mot_phia(0.95, 0.94)                       # thấy mức giảm 1 điểm từ 95 phần trăm, giả định
for f in (CANARY_NHO, CANARY_LON):
    lam = f * YEU_CAU_MOI_GIAY
    p99 = np.log(100) / (SUC_CHUA - lam) * 1e3 if lam < SUC_CHUA else np.inf
    print(f"canary {f:.0%}: {f * NHAN_MOI_GIO:,.0f} nhãn mỗi giờ → {can / (f * NHAN_MOI_GIO):5.1f} giờ để thấy mức giảm 1 điểm;"
          f" mức sử dụng phụ trợ {lam / SUC_CHUA:.0%}, độ trễ phân vị 99 ở phụ trợ {p99:.0f} ms")

In [ ]:
cac_f = np.linspace(0.01, 0.31, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(100 * cac_f, can / (cac_f * NHAN_MOI_GIO), color=MAU[0], lw=2.2)
ax1.set_yscale("log")
ax1.set_xlabel("phần lưu lượng của canary (%)")
ax1.set_ylabel("giờ để thấy mức giảm 1 điểm (thang log)")
ax1.set_title("Bằng chứng: canary nhỏ thì chờ lâu")
lam = cac_f * YEU_CAU_MOI_GIAY
ax2.plot(100 * cac_f[lam < SUC_CHUA * 0.995], (np.log(100) / (SUC_CHUA - lam) * 1e3)[lam < SUC_CHUA * 0.995],
         color=MAU[1], lw=2.2)
ax2.axvline(100 * SUC_CHUA / YEU_CAU_MOI_GIAY, color=XAM, lw=0.9, ls="--")
ax2.text(100 * SUC_CHUA / YEU_CAU_MOI_GIAY - 0.6, 1400, "hết sức chứa\n(giả định)", fontsize=8, color="0.3", ha="right")
ax2.set_ylim(0, 1600)
ax2.set_xlabel("phần lưu lượng của canary (%)")
ax2.set_ylabel("độ trễ phân vị 99 ở phụ trợ (ms)")
ax2.set_title("Tải: phụ trợ của mô hình mới, hàng đợi M/M/1")
for ax in (ax1, ax2):
    for f in (CANARY_NHO, CANARY_LON):
        ax.axvline(100 * f, color="0.2", lw=0.8, ls=":")
ax1.text(100 * CANARY_NHO + 0.4, ax1.get_ylim()[1] * 0.6, "5 %", fontsize=8)
ax1.text(100 * CANARY_LON - 0.4, ax1.get_ylim()[1] * 0.6, "30 %", fontsize=8, ha="right")
plt.tight_layout()
plt.show()

Hai lý do kéo về hai phía. Canary nhỏ an toàn về tải nhưng chậm về bằng chứng; canary lớn thấy nhanh nhưng có
thể đẩy một phụ trợ qua ngưỡng mà ở 5 phần trăm nó không bao giờ chạm tới. Độ trễ của hàng đợi tăng vọt
khi mức sử dụng tiến tới 1, nên một vấn đề "chỉ lộ ở 30 phần trăm" không mâu thuẫn gì với một lần chạy 5 phần
trăm sạch sẽ. Đó là lý do lưu lượng được nâng dần, theo từng bậc, có quay lại phiên bản trước (rollback) đã thử.

**Ngân sách độ trễ.** Bảng 14 chia một mức mục tiêu 100 ms ở phân vị 99 cho sáu thành phần. Suy luận chiếm
45 phần trăm. Một mô hình nhanh gấp đôi đưa độ trễ từ 100 xuống 77.5 ms, tức chỉ nhanh hơn 1.3 lần toàn trình.

In [ ]:
NGAN_SACH_MS = sach(100, nguon="100 ms · 15 · 25 · 5 · 45 · 5 · 5 ms")
THANH_PHAN = {"mạng, khứ hồi": sach(15, nguon="100 ms · 15 · 25"), "lấy đặc trưng": sach(25, nguon="15 · 25"),
              "phân tích yêu cầu": sach(5, nguon="25 · 5"), "suy luận": sach(45, nguon="5 · 45"),
              "hậu xử lý": sach(5, nguon="5 ms · 45 phần trăm"), "tuần tự hoá phản hồi": sach(5, nguon="5 ms · 45 phần trăm")}
assert sum(THANH_PHAN.values()) == NGAN_SACH_MS
NHANH_GAP = sach(2, nguon="45 phần trăm · 2 lần")


def toan_trinh(tang_toc, thanh_phan="suy luận"):
    """Độ trễ toàn trình khi một thành phần nhanh lên tang_toc lần (định luật Amdahl)."""
    return sum(v / tang_toc if k == thanh_phan else v for k, v in THANH_PHAN.items())


theo_sach("độ trễ khi suy luận nhanh gấp đôi (ms)", toan_trinh(NHANH_GAP), sach=77.5, nguon="77.5 ms")
theo_sach("tăng tốc toàn trình (lần)", NGAN_SACH_MS / toan_trinh(NHANH_GAP), sach=1.3, nguon="1.3 lần")
print(f"suy luận nhanh vô hạn: {toan_trinh(np.inf):.0f} ms, tăng tốc tối đa {NGAN_SACH_MS / toan_trinh(np.inf):.2f} lần;"
      f" phần độ trễ giảm được tối đa {1 - toan_trinh(np.inf) / NGAN_SACH_MS:.0%},"
      f" dưới {sach(50, nguon='dưới 50 phần trăm')} phần trăm")
print(f"lấy đặc trưng nhanh gấp đôi: {toan_trinh(2, 'lấy đặc trưng'):.1f} ms")

In [ ]:
cac_tang_toc = np.logspace(0, 2, 200)
fig, ax = plt.subplots(figsize=(8, 3.8))
for ten, mau in (("suy luận", MAU[0]), ("lấy đặc trưng", MAU[1])):
    ax.plot(cac_tang_toc, [toan_trinh(s, ten) for s in cac_tang_toc], color=mau, lw=2.2)
    ax.text(100, toan_trinh(100, ten) + 2, f"chỉ tăng tốc {ten}", fontsize=8, ha="right", color="0.2")
ax.plot([NHANH_GAP], [toan_trinh(NHANH_GAP)], "o", color=MAU[0], ms=8, zorder=5)
ax.annotate("bảng 14: suy luận gấp 2 → 77.5 ms", (NHANH_GAP, toan_trinh(NHANH_GAP)), xytext=(10, -16),
            textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_ylim(0, 105)
ax.set_xlabel("một thành phần nhanh lên bao nhiêu lần (thang log)")
ax.set_ylabel("độ trễ phân vị 99 toàn trình (ms)")
ax.set_title("Ngân sách độ trễ của bảng 14: phần không tăng tốc thành sàn")
plt.tight_layout()
plt.show()

**Ca Oura.** Mô hình chỉ dùng gia tốc kế phân loại bốn giai đoạn ngủ đúng 57 phần trăm, mô hình có thêm đặc
trưng hệ thần kinh tự chủ và nhịp ngày đêm đạt 79 phần trăm, trong khi độ đồng thuận giữa hai chuyên gia chấm
đa ký giấc ngủ là khoảng 82 tới 83 phần trăm. Sách nói mức tăng 22 điểm khép được khoảng 84.6 tới 88 phần trăm
khoảng cách từ mức gốc tới mức đồng thuận của con người, và nhắc rằng độ đồng thuận ấy không phải trần của độ
chính xác.

In [ ]:
GOC = sach(57, nguon="57 phần trăm · 79 phần trăm")
MOI = sach(79, nguon="57 phần trăm · 79 phần trăm")
NGUOI_THAP = sach(82, nguon="82–83 phần trăm")
NGUOI_CAO = sach(83, nguon="82–83 phần trăm")
theo_sach("mức tăng (điểm)", MOI - GOC, sach=22, nguon="22 điểm")
theo_sach("phần khoảng cách đã khép, so với 83 (phần trăm)", 100 * (MOI - GOC) / (NGUOI_CAO - GOC), sach=84.6,
          nguon="84.6–88 phần trăm")
theo_sach("phần khoảng cách đã khép, so với 82 (phần trăm)", 100 * (MOI - GOC) / (NGUOI_THAP - GOC), sach=88,
          nguon="84.6–88 phần trăm")

## Thử thêm

1. Ở phần 1, đổi tỉ lệ điểm ảnh bị mất từ `0.20` thành `0.05` rồi `0.50`. Cái giá của lệch giữa điền trung
   bình và điền 0 lớn lên thế nào, và cổng KS còn chặn được ở `0.05` không?
2. Ở phần 4, đặt `C=0.2` cho `A_moi` thay vì `1.0`, một thay đổi nhỏ hơn. Hệ thống A mới cộng B cũ còn kém hơn
   A mới một mình không?
3. Ở phần 6, đặt `SUC_CHUA = 60`. Canary 5 phần trăm còn an toàn về tải không? Rồi đặt `NHAN_MOI_GIO = 200` và
   xem canary 5 phần trăm phải chạy bao nhiêu giờ.

## Tóm lại

* Lệch giữa huấn luyện và phục vụ đổi độ chính xác mà không đổi một dòng mã mô hình: một hằng số chuẩn hoá,
  một cách điền giá trị thiếu, một tham số mặc định của thư viện.
* Một cổng KS trong CI bắt được lệch làm đổi phân phối, nhưng cho qua hai cột cùng phân phối bị đảo chỗ; phép
  kiểm ngang bằng trên cùng những dòng mới bắt được loại lỗi đó.
* Một đặc trưng không đúng theo mốc thời gian làm độ chính xác ngoại tuyến đẹp giả, và không phép so phân phối
  nào trên dữ liệu phục vụ thấy được.
* Trong một thác hiệu chỉnh, cải thiện mô hình đầu chuỗi có thể làm hệ thống kém đi cho tới khi mọi mô hình phía
  sau được học lại.
* Phương trình 9 cho khoảng 745,644 người mỗi nhánh cho mức tăng 0.1 điểm; nhìn trộm nhiều lần làm báo nhầm
  tăng gấp mấy lần; canary nhỏ chậm về bằng chứng, canary lớn có thể quá tải, và suy luận nhanh gấp đôi chỉ
  cho 1.3 lần toàn trình.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Vận hành học máy — khi hệ thống sẵn sàng mà vẫn sai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch14-ml-ops/), dựng từ chương
[*ML Operations*](https://mlsysbook.ai/vol1/ml_ops/ml_ops.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_14_ml_ops.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.